# Stochastic Relay Encoders

**High-level idea:** Create additional encoders that "relay" representations forward through
additional stochastic latent spaces. The repeated stochasticity encourages *partial hardness* —
something we want to characterize and then use to our advantage.

## Architecture Overview

1. **Encoder** $f_\theta(x, \epsilon)$: Maps input $X$ to a probabilistic latent $p(u_0|x)$
2. **Relay** $g(u_{i-1}, \epsilon')$: Maps latent $u_{i-1}$ to next stochastic latent $p(u_i|u_{i-1})$, weight-shared across $k$ relays
3. **Decoder**: Reconstructs $\hat{X}$ from the final relay output $u_k$

### Loss
$$\mathcal{L}_{\text{relay}} = \mathcal{L}_{\text{recon}} + \frac{\beta}{k+1}\left[D_{\text{KL}}(p(u_0|x)\|r(u)) + \sum_{i=1}^{k} D_{\text{KL}}(p(u_i|u_{i-1})\|r(u_i))\right]$$

In [ ]:
# ============================================================
# Step 0: Imports and Configuration
# ============================================================
# We use PyTorch for the neural network components.
# MNIST is used as a simple benchmark to validate the relay encoder.

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# ============================================================
# Step 1: Hyperparameters
# ============================================================
# - latent_dim: dimensionality of each stochastic latent space u_i
# - num_relays (k): how many relay steps to propagate through
# - beta: coefficient scaling the KL divergence penalty
#   * beta >> 1 → vacuous representations (all posteriors collapse to prior)
#   * beta → 0  → classic autoencoder (no information penalty)
# - beta is divided by (k+1) in the loss so its effect scales
#   consistently regardless of the number of relays.

LATENT_DIM = 20
NUM_RELAYS = 3        # k relay steps
BETA = 4.0            # KL weight (before normalization by k+1)
HIDDEN_DIM = 400
BATCH_SIZE = 128
EPOCHS = 20
LR = 1e-3
BETA_ANNEAL = True    # whether to ramp beta upward over training
INPUT_DIM = 784       # 28x28 MNIST images flattened

In [ ]:
# ============================================================
# Step 2: Data Loading
# ============================================================
# X ~ p(x): MNIST digits as our input distribution.
# Think of X as data collected by a remote sensor in need of
# compression before transmission to a central processor.

transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")

In [ ]:
# ============================================================
# Step 3: Reparameterization Trick
# ============================================================
# To backpropagate through the stochastic sampling step, we use
# the reparameterization trick:
#   u = mu + sigma * epsilon,  where epsilon ~ N(0, I)
# This makes the stochasticity external to the parameters,
# allowing gradient flow through mu and sigma.

def reparameterize(mu, logvar):
    """Sample from N(mu, sigma^2) using the reparameterization trick.
    
    Args:
        mu: Mean of the distribution (batch_size, latent_dim)
        logvar: Log variance of the distribution (batch_size, latent_dim)
    
    Returns:
        Sampled point u = mu + sigma * epsilon
    """
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)  # epsilon ~ N(0, I)
    return mu + std * eps

In [ ]:
# ============================================================
# Step 4: KL Divergence Computation
# ============================================================
# The KL divergence between the posterior p(u|x) = N(mu, sigma^2)
# and the prior r(u) = N(0, I) provides an upper bound on the
# mutual information I(X; U):
#   I(X; U) <= E_{x~p(x)} [D_KL(p(u|x) || r(u))]
#
# Closed-form for diagonal Gaussians:
#   D_KL = -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)

def kl_divergence(mu, logvar):
    """Compute KL(N(mu, sigma^2) || N(0, I)).
    
    Returns per-sample KL divergence (batch_size,).
    """
    return -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=-1)

In [ ]:
# ============================================================
# Step 5: Encoder Network f_theta(x, epsilon)
# ============================================================
# The encoder compresses input x into a probabilistic
# representation p(u_0 | x) parameterized by (mu_0, logvar_0).
# Stochasticity is introduced via epsilon in the reparameterization
# trick. This is the first stage of compression.

class Encoder(nn.Module):
    """Maps input x to probabilistic latent p(u_0|x).
    
    This is the initial compression stage: x -> (mu_0, logvar_0).
    A point u_0 is then sampled from N(mu_0, sigma_0^2).
    """
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
    
    def forward(self, x):
        h = F.relu(self.fc1(x))
        h = F.relu(self.fc2(h))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

In [ ]:
# ============================================================
# Step 6: Relay Network g(u_{i-1}, epsilon')
# ============================================================
# The relay takes a sampled latent point u_{i-1} and maps it to
# another probabilistic latent space p(u_i | u_{i-1}).
#
# KEY DESIGN CHOICES:
# - Weight sharing: the SAME relay network is applied k times
#   in sequence (u_0 -> u_1 -> ... -> u_k). This is analogous
#   to a recurrent application of the same transformation.
# - Each relay introduces fresh stochasticity epsilon', which
#   encourages "partial hardness" in the representations.
# - The relay operates in latent space (latent_dim -> latent_dim),
#   NOT in data space.

class RelayEncoder(nn.Module):
    """Maps latent u_{i-1} to probabilistic latent p(u_i | u_{i-1}).
    
    Weight-shared across all k relay steps. Each application
    introduces a new KL penalty and fresh stochastic sampling.
    """
    def __init__(self, latent_dim, hidden_dim):
        super().__init__()
        self.fc1 = nn.Linear(latent_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
    
    def forward(self, u):
        h = F.relu(self.fc1(u))
        h = F.relu(self.fc2(h))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

In [ ]:
# ============================================================
# Step 7: Decoder Network
# ============================================================
# The decoder maps the final relay output u_k back to data space
# to produce the reconstruction x_hat. The reconstruction loss
# L_recon measures fidelity (here, binary cross-entropy for
# pixel-valued images; could also be Euclidean/MSE).

class Decoder(nn.Module):
    """Maps final latent u_k back to data space x_hat."""
    def __init__(self, latent_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(latent_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, u):
        h = F.relu(self.fc1(u))
        h = F.relu(self.fc2(h))
        return torch.sigmoid(self.fc_out(h))

In [ ]:
# ============================================================
# Step 8: Full Stochastic Relay VAE Model
# ============================================================
# Assembles the full pipeline:
#   x -> Encoder -> p(u_0|x) -> sample u_0
#     -> Relay -> p(u_1|u_0) -> sample u_1
#     -> Relay -> p(u_2|u_1) -> sample u_2  (same weights)
#     -> ...  (k times total)
#     -> Decoder -> x_hat
#
# The forward pass collects all (mu, logvar) pairs to compute
# the full relay loss with KL terms from each stochastic layer.

class StochasticRelayVAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, num_relays):
        super().__init__()
        self.num_relays = num_relays
        
        # f_theta: initial encoder x -> p(u_0|x)
        self.encoder = Encoder(input_dim, hidden_dim, latent_dim)
        
        # g: relay encoder u_{i-1} -> p(u_i|u_{i-1})
        # Weight-shared across all k relay steps
        self.relay = RelayEncoder(latent_dim, hidden_dim)
        
        # Decoder: u_k -> x_hat
        self.decoder = Decoder(latent_dim, hidden_dim, input_dim)
    
    def forward(self, x):
        # --- Initial encoding: x -> p(u_0|x) ---
        mu_0, logvar_0 = self.encoder(x)
        u = reparameterize(mu_0, logvar_0)
        
        # Collect all KL distribution parameters for the loss
        all_mu = [mu_0]
        all_logvar = [logvar_0]
        
        # --- Relay steps: u_{i-1} -> p(u_i|u_{i-1}) for i=1..k ---
        # The SAME relay network (weight sharing) is applied k times.
        # Each step introduces fresh stochasticity via reparameterization.
        for _ in range(self.num_relays):
            mu_i, logvar_i = self.relay(u)
            u = reparameterize(mu_i, logvar_i)
            all_mu.append(mu_i)
            all_logvar.append(logvar_i)
        
        # --- Decode: u_k -> x_hat ---
        x_hat = self.decoder(u)
        
        return x_hat, all_mu, all_logvar

In [ ]:
# ============================================================
# Step 9: Relay Loss Function
# ============================================================
# The full loss for the stochastic relay VAE:
#
#   L_relay = L_recon + (beta / (k+1)) * [
#       D_KL(p(u_0|x) || r(u))
#       + sum_{i=1}^{k} D_KL(p(u_i|u_{i-1}) || r(u_i))
#   ]
#
# The normalization by (k+1) ensures that beta has a consistent
# effect regardless of how many relays we use. Without this,
# adding more relays would implicitly increase the total KL
# penalty and push representations toward the prior.
#
# L_recon: Binary cross-entropy (pixel-level reconstruction)
# Could also use MSE for Euclidean distance in pixel space.

def relay_loss(x, x_hat, all_mu, all_logvar, beta, num_relays):
    """Compute the stochastic relay VAE loss.
    
    Args:
        x: Original input (batch_size, input_dim)
        x_hat: Reconstruction (batch_size, input_dim)
        all_mu: List of mu tensors from encoder + each relay
        all_logvar: List of logvar tensors from encoder + each relay
        beta: KL weight coefficient
        num_relays: Number of relay steps k
    
    Returns:
        total_loss, recon_loss, total_kl (all scalar)
    """
    # Reconstruction loss: -E[log p(x|u_k)]
    recon_loss = F.binary_cross_entropy(x_hat, x, reduction='sum') / x.size(0)
    
    # Sum KL divergences from all stochastic layers:
    # D_KL(p(u_0|x)||r(u)) + sum_i D_KL(p(u_i|u_{i-1})||r(u_i))
    total_kl = torch.zeros(1, device=x.device)
    for mu, logvar in zip(all_mu, all_logvar):
        total_kl += kl_divergence(mu, logvar).mean()
    
    # Scale by beta/(k+1) so beta has consistent meaning across
    # different numbers of relays
    scaled_kl = (beta / (num_relays + 1)) * total_kl
    
    total_loss = recon_loss + scaled_kl
    return total_loss, recon_loss, total_kl

In [ ]:
# ============================================================
# Step 10: Beta Annealing Schedule
# ============================================================
# Optionally ramp beta upward over training (logarithmic schedule).
# This helps early training focus on reconstruction quality before
# gradually increasing the information bottleneck pressure.
# Without annealing, high beta can cause posterior collapse early
# on, where the encoder ignores the input and outputs the prior.

def get_beta(epoch, max_beta, total_epochs, anneal=True):
    """Compute beta for current epoch.
    
    Logarithmic annealing: beta ramps from near 0 to max_beta.
    """
    if not anneal:
        return max_beta
    # Logarithmic ramp: slow start, accelerating increase
    progress = epoch / max(total_epochs - 1, 1)
    return max_beta * (np.log1p(progress * (np.e - 1)) / 1.0)

In [ ]:
# ============================================================
# Step 11: Initialize Model and Optimizer
# ============================================================

model = StochasticRelayVAE(
    input_dim=INPUT_DIM,
    hidden_dim=HIDDEN_DIM,
    latent_dim=LATENT_DIM,
    num_relays=NUM_RELAYS,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# Print model architecture to verify weight sharing:
# The relay encoder appears once, but is applied k times in forward()
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")
print(f"Relay steps (k): {NUM_RELAYS}")
print(f"Note: Relay encoder weights are shared across all {NUM_RELAYS} steps")

In [ ]:
# ============================================================
# Step 12: Training Loop
# ============================================================
# End-to-end training of the full pipeline:
#   Encoder + k shared Relays + Decoder
#
# We track per-relay KL divergences to observe how information
# flows through the stochastic relay chain. Ideally, relays
# will show "partial hardness" — some information passes through
# each relay but is progressively compressed.

history = {
    'train_loss': [], 'train_recon': [], 'train_kl': [],
    'test_loss': [], 'test_recon': [], 'test_kl': [],
    'beta_values': [],
    'per_layer_kl': [],  # Track KL at each stochastic layer
}

for epoch in range(EPOCHS):
    # --- Beta annealing ---
    current_beta = get_beta(epoch, BETA, EPOCHS, anneal=BETA_ANNEAL)
    history['beta_values'].append(current_beta)
    
    # === Training ===
    model.train()
    train_loss_sum, train_recon_sum, train_kl_sum = 0, 0, 0
    epoch_layer_kls = [0.0] * (NUM_RELAYS + 1)  # encoder + k relays
    n_batches = 0
    
    for x_batch, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False):
        x_batch = x_batch.view(-1, INPUT_DIM).to(device)
        
        # Forward pass through encoder -> relays -> decoder
        x_hat, all_mu, all_logvar = model(x_batch)
        
        # Compute relay loss: L_recon + (beta/(k+1)) * sum(KL_i)
        loss, recon, kl = relay_loss(
            x_batch, x_hat, all_mu, all_logvar,
            current_beta, NUM_RELAYS
        )
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss_sum += loss.item()
        train_recon_sum += recon.item()
        train_kl_sum += kl.item()
        
        # Track per-layer KL to analyze information flow
        for j, (mu, logvar) in enumerate(zip(all_mu, all_logvar)):
            epoch_layer_kls[j] += kl_divergence(mu, logvar).mean().item()
        n_batches += 1
    
    # Average over batches
    history['train_loss'].append(train_loss_sum / n_batches)
    history['train_recon'].append(train_recon_sum / n_batches)
    history['train_kl'].append(train_kl_sum / n_batches)
    history['per_layer_kl'].append([kl / n_batches for kl in epoch_layer_kls])
    
    # === Evaluation ===
    model.eval()
    test_loss_sum, test_recon_sum, test_kl_sum = 0, 0, 0
    n_test = 0
    
    with torch.no_grad():
        for x_batch, _ in test_loader:
            x_batch = x_batch.view(-1, INPUT_DIM).to(device)
            x_hat, all_mu, all_logvar = model(x_batch)
            loss, recon, kl = relay_loss(
                x_batch, x_hat, all_mu, all_logvar,
                current_beta, NUM_RELAYS
            )
            test_loss_sum += loss.item()
            test_recon_sum += recon.item()
            test_kl_sum += kl.item()
            n_test += 1
    
    history['test_loss'].append(test_loss_sum / n_test)
    history['test_recon'].append(test_recon_sum / n_test)
    history['test_kl'].append(test_kl_sum / n_test)
    
    # Print epoch summary with per-layer KL breakdown
    layer_kl_str = " | ".join(
        [f"KL_u{i}={history['per_layer_kl'][-1][i]:.1f}" for i in range(NUM_RELAYS + 1)]
    )
    print(
        f"Epoch {epoch+1:2d} | β={current_beta:.2f} | "
        f"Train: loss={history['train_loss'][-1]:.1f} recon={history['train_recon'][-1]:.1f} "
        f"kl={history['train_kl'][-1]:.1f} | Test: loss={history['test_loss'][-1]:.1f} | "
        f"{layer_kl_str}"
    )

In [ ]:
# ============================================================
# Step 13: Visualize Training Curves
# ============================================================
# Plot reconstruction loss, total KL, and beta schedule to
# understand the rate-distortion tradeoff during training.

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

# (a) Total loss
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['test_loss'], label='Test')
axes[0].set_title('Total Loss (L_relay)')
axes[0].set_xlabel('Epoch')
axes[0].legend()

# (b) Reconstruction loss
axes[1].plot(history['train_recon'], label='Train')
axes[1].plot(history['test_recon'], label='Test')
axes[1].set_title('Reconstruction Loss (L_recon)')
axes[1].set_xlabel('Epoch')
axes[1].legend()

# (c) Total KL divergence
axes[2].plot(history['train_kl'], label='Train')
axes[2].plot(history['test_kl'], label='Test')
axes[2].set_title('Total KL Divergence')
axes[2].set_xlabel('Epoch')
axes[2].legend()

# (d) Beta annealing schedule
axes[3].plot(history['beta_values'], 'r-')
axes[3].set_title('β Annealing Schedule')
axes[3].set_xlabel('Epoch')
axes[3].set_ylabel('β')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Step 14: Per-Layer KL Analysis (Characterizing Partial Hardness)
# ============================================================
# This is the KEY analysis for understanding stochastic relays.
#
# "Partial hardness" means that each relay stage transmits SOME
# but not ALL information. By plotting the KL divergence at each
# layer over training, we can see:
# - How much information each relay layer transmits
# - Whether later relays become "harder" (lower KL = less info)
# - The information gradient across the relay chain
#
# If KL_u0 > KL_u1 > ... > KL_uk, information is progressively
# compressed — each relay strips away more detail.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Per-layer KL over training epochs
per_layer_kl_array = np.array(history['per_layer_kl'])
for i in range(NUM_RELAYS + 1):
    label = f"Encoder (u₀)" if i == 0 else f"Relay {i} (u_{i})"
    axes[0].plot(per_layer_kl_array[:, i], label=label)
axes[0].set_title('Per-Layer KL Divergence Over Training')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('KL Divergence')
axes[0].legend()

# (b) Final epoch: KL at each layer (information flow profile)
final_kls = per_layer_kl_array[-1]
layer_names = ['Encoder\n(u₀)'] + [f'Relay {i}\n(u_{i})' for i in range(1, NUM_RELAYS + 1)]
axes[1].bar(layer_names, final_kls, color=['steelblue'] + ['coral'] * NUM_RELAYS)
axes[1].set_title('Information Flow Profile (Final Epoch)')
axes[1].set_ylabel('KL Divergence')
axes[1].set_xlabel('Stochastic Layer')

plt.tight_layout()
plt.show()

print("\nPartial Hardness Analysis:")
print("="*50)
for i, kl_val in enumerate(final_kls):
    layer = "Encoder (u₀)" if i == 0 else f"Relay {i} (u_{i})"
    print(f"  {layer}: KL = {kl_val:.2f}")
print(f"  Total KL: {sum(final_kls):.2f}")
print(f"  Avg KL per layer: {sum(final_kls)/(NUM_RELAYS+1):.2f}")

In [ ]:
# ============================================================
# Step 15: Reconstruction Visualization
# ============================================================
# Show original images and their reconstructions to qualitatively
# assess how well information survives the relay chain.

model.eval()
with torch.no_grad():
    x_sample, _ = next(iter(test_loader))
    x_sample = x_sample[:16].view(-1, INPUT_DIM).to(device)
    x_hat, _, _ = model(x_sample)

fig, axes = plt.subplots(2, 16, figsize=(16, 2.5))
for i in range(16):
    axes[0, i].imshow(x_sample[i].cpu().view(28, 28), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(x_hat[i].cpu().view(28, 28), cmap='gray')
    axes[1, i].axis('off')
axes[0, 0].set_ylabel('Original', fontsize=10)
axes[1, 0].set_ylabel('Recon', fontsize=10)
plt.suptitle(f'Reconstructions (k={NUM_RELAYS} relays, β={BETA})', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Step 16: Intermediate Relay Reconstructions
# ============================================================
# Decode from each intermediate relay stage to visualize
# progressive information loss through the relay chain.
# This directly shows the "partial hardness" effect:
# reconstructions from later relays should be progressively
# more blurred or abstract.

model.eval()
with torch.no_grad():
    x_sample, _ = next(iter(test_loader))
    x_single = x_sample[0:1].view(-1, INPUT_DIM).to(device)
    
    # Manually propagate through encoder and each relay,
    # decoding at each stage
    mu_0, logvar_0 = model.encoder(x_single)
    u = reparameterize(mu_0, logvar_0)
    
    intermediate_recons = []
    intermediate_recons.append(model.decoder(u))  # Decode from u_0
    
    for i in range(NUM_RELAYS):
        mu_i, logvar_i = model.relay(u)
        u = reparameterize(mu_i, logvar_i)
        intermediate_recons.append(model.decoder(u))  # Decode from u_i

fig, axes = plt.subplots(1, NUM_RELAYS + 2, figsize=(3 * (NUM_RELAYS + 2), 3))
axes[0].imshow(x_single.cpu().view(28, 28), cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

for i, recon in enumerate(intermediate_recons):
    axes[i + 1].imshow(recon.cpu().view(28, 28), cmap='gray')
    axes[i + 1].set_title(f'From u_{i}')
    axes[i + 1].axis('off')

plt.suptitle('Decoding from Each Relay Stage (Partial Hardness Visualization)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Step 17: Latent Space Visualization (t-SNE)
# ============================================================
# Visualize how the latent space structure changes across relay
# stages. The prior r(u) = N(0,I) is a single Gaussian blob.
# With increasing relays, representations should become
# progressively more "mixed" / closer to the prior.

from sklearn.manifold import TSNE

model.eval()
all_latents = [[] for _ in range(NUM_RELAYS + 1)]
all_labels = []

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.view(-1, INPUT_DIM).to(device)
        
        # Propagate through encoder
        mu_0, logvar_0 = model.encoder(x_batch)
        u = reparameterize(mu_0, logvar_0)
        all_latents[0].append(u.cpu())
        
        # Propagate through relays
        for i in range(NUM_RELAYS):
            mu_i, logvar_i = model.relay(u)
            u = reparameterize(mu_i, logvar_i)
            all_latents[i + 1].append(u.cpu())
        
        all_labels.append(y_batch)

all_labels = torch.cat(all_labels).numpy()

# Subsample for t-SNE speed
n_vis = 3000
idx = np.random.choice(len(all_labels), n_vis, replace=False)

fig, axes = plt.subplots(1, NUM_RELAYS + 1, figsize=(5 * (NUM_RELAYS + 1), 4.5))
for stage in range(NUM_RELAYS + 1):
    latents = torch.cat(all_latents[stage]).numpy()[idx]
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    coords = tsne.fit_transform(latents)
    
    scatter = axes[stage].scatter(
        coords[:, 0], coords[:, 1],
        c=all_labels[idx], cmap='tab10', s=5, alpha=0.6
    )
    title = 'Encoder (u₀)' if stage == 0 else f'Relay {stage} (u_{stage})'
    axes[stage].set_title(title)
    axes[stage].set_xticks([])
    axes[stage].set_yticks([])

plt.suptitle('Latent Space Structure Across Relay Stages (t-SNE)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Step 18: Comparison Experiment — Vary Number of Relays
# ============================================================
# Train models with k=0 (standard beta-VAE), k=1, k=3, k=5
# relays and compare their rate-distortion tradeoffs.
# This characterizes how relay depth affects partial hardness.

def train_model(num_relays, epochs=15, beta=4.0):
    """Train a StochasticRelayVAE with given number of relays."""
    m = StochasticRelayVAE(INPUT_DIM, HIDDEN_DIM, LATENT_DIM, num_relays).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=LR)
    
    results = {'recon': [], 'kl': [], 'per_layer_kl': []}
    
    for epoch in range(epochs):
        current_beta = get_beta(epoch, beta, epochs, anneal=True)
        m.train()
        for x_batch, _ in train_loader:
            x_batch = x_batch.view(-1, INPUT_DIM).to(device)
            x_hat, all_mu, all_logvar = m(x_batch)
            loss, _, _ = relay_loss(x_batch, x_hat, all_mu, all_logvar, current_beta, num_relays)
            opt.zero_grad()
            loss.backward()
            opt.step()
        
        # Evaluate
        m.eval()
        test_recon, test_kl, n = 0, 0, 0
        layer_kls = [0.0] * (num_relays + 1)
        with torch.no_grad():
            for x_batch, _ in test_loader:
                x_batch = x_batch.view(-1, INPUT_DIM).to(device)
                x_hat, all_mu, all_logvar = m(x_batch)
                _, recon, kl = relay_loss(x_batch, x_hat, all_mu, all_logvar, current_beta, num_relays)
                test_recon += recon.item()
                test_kl += kl.item()
                for j, (mu, lv) in enumerate(zip(all_mu, all_logvar)):
                    layer_kls[j] += kl_divergence(mu, lv).mean().item()
                n += 1
        results['recon'].append(test_recon / n)
        results['kl'].append(test_kl / n)
        results['per_layer_kl'].append([k / n for k in layer_kls])
    
    return m, results

# Train models with different relay counts
relay_configs = [0, 1, 3, 5]
comparison = {}

for k in relay_configs:
    print(f"\nTraining with k={k} relays...")
    m, r = train_model(k, epochs=15)
    comparison[k] = r
    print(f"  Final — Recon: {r['recon'][-1]:.1f}, KL: {r['kl'][-1]:.1f}")

In [ ]:
# ============================================================
# Step 19: Comparison Visualization
# ============================================================
# Plot rate (KL) vs distortion (reconstruction loss) across
# different relay depths. More relays should shift the tradeoff.

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Reconstruction loss across relay counts
for k in relay_configs:
    label = f'k={k} ({"VAE" if k == 0 else "relays"})'
    axes[0].plot(comparison[k]['recon'], label=label)
axes[0].set_title('Reconstruction Loss vs Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Reconstruction Loss')
axes[0].legend()

# (b) Total KL across relay counts
for k in relay_configs:
    label = f'k={k} ({"VAE" if k == 0 else "relays"})'
    axes[1].plot(comparison[k]['kl'], label=label)
axes[1].set_title('Total KL Divergence vs Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Total KL')
axes[1].legend()

# (c) Rate-Distortion plot (final epoch)
for k in relay_configs:
    axes[2].scatter(
        comparison[k]['kl'][-1], comparison[k]['recon'][-1],
        s=100, label=f'k={k}', zorder=5
    )
axes[2].set_title('Rate-Distortion Tradeoff (Final Epoch)')
axes[2].set_xlabel('Rate (Total KL)')
axes[2].set_ylabel('Distortion (Recon Loss)')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Step 20: Per-Layer KL Profiles Across Relay Depths
# ============================================================
# For each model, show the KL at each stochastic layer.
# This is the "partial hardness profile" — how information
# distributes across the relay chain.

fig, axes = plt.subplots(1, len(relay_configs), figsize=(5 * len(relay_configs), 4))

for idx, k in enumerate(relay_configs):
    final_kls = comparison[k]['per_layer_kl'][-1]
    names = ['Enc'] + [f'R{i}' for i in range(1, k + 1)]
    colors = ['steelblue'] + ['coral'] * k
    axes[idx].bar(names, final_kls, color=colors)
    axes[idx].set_title(f'k={k} relays')
    axes[idx].set_ylabel('KL Divergence')
    axes[idx].set_xlabel('Layer')

plt.suptitle('Partial Hardness Profiles: Per-Layer KL at Different Relay Depths', y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Step 21: Sampling from the Prior (Generative Quality)
# ============================================================
# Sample u_k ~ r(u) = N(0, I) and decode.
# For relay models, we can also sample from the prior and
# propagate BACKWARDS through relays (though here we simply
# decode from a single prior sample, which the decoder was
# trained to handle from the u_k space).

model.eval()
with torch.no_grad():
    z = torch.randn(64, LATENT_DIM).to(device)
    samples = model.decoder(z)

fig, axes = plt.subplots(8, 8, figsize=(8, 8))
for i in range(64):
    r, c = i // 8, i % 8
    axes[r, c].imshow(samples[i].cpu().view(28, 28), cmap='gray')
    axes[r, c].axis('off')
plt.suptitle(f'Samples from Prior (k={NUM_RELAYS} relay model)', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Step 22: Variance Analysis — Measuring Stochastic Hardness
# ============================================================
# For a single input x, run it through the relay chain multiple
# times and measure the variance of the output at each stage.
# Higher variance = "softer" representation (more stochastic).
# Lower variance = "harder" (more deterministic).
# Partial hardness = intermediate variance, possibly varying
# across relay stages.

model.eval()
N_SAMPLES = 200

with torch.no_grad():
    x_single, _ = next(iter(test_loader))
    x_single = x_single[0:1].view(-1, INPUT_DIM).to(device)
    
    # Collect samples at each relay stage
    stage_samples = [[] for _ in range(NUM_RELAYS + 1)]
    
    for _ in range(N_SAMPLES):
        mu_0, logvar_0 = model.encoder(x_single)
        u = reparameterize(mu_0, logvar_0)
        stage_samples[0].append(u.cpu())
        
        for i in range(NUM_RELAYS):
            mu_i, logvar_i = model.relay(u)
            u = reparameterize(mu_i, logvar_i)
            stage_samples[i + 1].append(u.cpu())

# Compute per-stage variance (averaged over latent dimensions)
stage_variances = []
for stage in range(NUM_RELAYS + 1):
    samples_tensor = torch.cat(stage_samples[stage], dim=0)  # (N_SAMPLES, latent_dim)
    var = samples_tensor.var(dim=0).mean().item()  # avg variance across dims
    stage_variances.append(var)

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
layer_names = ['Encoder\n(u₀)'] + [f'Relay {i}\n(u_{i})' for i in range(1, NUM_RELAYS + 1)]
ax.bar(layer_names, stage_variances, color=['steelblue'] + ['coral'] * NUM_RELAYS)
ax.set_ylabel('Average Variance of Sampled Representations')
ax.set_title('Stochastic Hardness Across Relay Stages\n(Higher = Softer, Lower = Harder)')

for i, v in enumerate(stage_variances):
    ax.text(i, v + 0.01 * max(stage_variances), f'{v:.4f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## Summary

### What we built:
1. **Encoder** $f_\theta(x, \epsilon)$: Compresses input to first stochastic latent $p(u_0|x)$
2. **Relay** $g(u_{i-1}, \epsilon')$: Weight-shared network applied $k$ times, creating a chain $u_0 \to u_1 \to \ldots \to u_k$
3. **Decoder**: Reconstructs from $u_k$

### Key observations to look for:
- **Per-layer KL profiles**: Do later relays have lower KL? This indicates progressive information compression.
- **Variance analysis**: Does variance change across relay stages? This characterizes partial hardness.
- **Rate-distortion tradeoff**: How does adding relays shift the balance between compression and reconstruction?
- **Latent space structure**: Do later relay stages show more "mixed" clusters (closer to the prior)?

### Design choices:
- **Weight sharing** across relays keeps parameter count constant regardless of $k$
- **$\beta/(k+1)$ normalization** ensures consistent KL scaling across relay depths
- **Beta annealing** prevents early posterior collapse